# Análisis Exploratorio de Datos (EDA) — GasHisSDB

**TFM: Clasificación binaria (Normal / Abnormal) de histopatología de cáncer gástrico**

Este cuaderno realiza un análisis exploratorio profundo del dataset **GasHisSDB**
(*Gastric Histopathology Sub-size Image Database*), previo al diseño del pipeline de
clasificación. El dataset contiene parches recortados de imágenes de portaobjetos
completas (WSI, *Whole Slide Images*) teñidas con hematoxilina-eosina (H&E), en tres
resoluciones (80×80, 120×120 y 160×160 px) y dos clases (`Normal`, `Abnormal`).

El objetivo del EDA no es solo describir el dataset, sino **detectar riesgos
metodológicos** que condicionan la validez de los resultados posteriores:

- Balance de clases y su impacto en la estrategia de entrenamiento y métricas.
- **Fuga de datos (*data leakage*) a nivel de WSI** — el riesgo más crítico en
  histopatología de parches.
- Duplicados / *near-duplicates* que inflarían artificialmente el rendimiento.
- Diferencias sistemáticas de color/tinción entre clases que el modelo podría explotar
  como **atajo espurio** (*shortcut learning*) en lugar de aprender morfología real.
- Estadísticas de normalización específicas del dominio (frente a las de ImageNet).

Todas las figuras se muestran *inline* y se guardan además en `results/eda/`.


## 1. Configuración del entorno

Importamos las librerías, fijamos la semilla global (**42**) para reproducibilidad y
definimos las rutas. El cuaderno está pensado para ejecutarse desde la **raíz del
proyecto** (que dentro del contenedor Docker es `/workspace`).

In [5]:
import os
import re
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import yaml

try:
    import imagehash
    HAS_IMAGEHASH = True
except ImportError:
    HAS_IMAGEHASH = False

%matplotlib inline
warnings.filterwarnings("ignore")

# --- Reproducibilidad ---
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# --- Estética ---
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110

# --- Rutas (relativas a la raíz del proyecto = /workspace en Docker) ---
PROJECT_ROOT = Path.cwd()
DATASET_ROOT = Path("dataset/GasHisSDB/GasHisSDB")
RESULTS_DIR  = Path("results")
EDA_DIR      = RESULTS_DIR / "eda"
CONFIGS_DIR  = Path("configs")
for d in (EDA_DIR, CONFIGS_DIR):
    d.mkdir(parents=True, exist_ok=True)

RESOLUTIONS    = ["80", "120", "160"]
CLASSES        = ["Normal", "Abnormal"]
CLASS_COLORS   = {"Normal": "#4C72B0", "Abnormal": "#C44E52"}
EXPECTED_TOTAL = 245196

assert DATASET_ROOT.exists(), f"No existe {DATASET_ROOT.resolve()} — revisa el bind mount de Docker."

print("Raíz del proyecto :", PROJECT_ROOT)
print("Dataset           :", DATASET_ROOT.resolve())
print("NumPy / Pandas    :", np.__version__, "/", pd.__version__)
print("imagehash disp.   :", HAS_IMAGEHASH)


def list_images(res, cls):
    '''Devuelve la lista ordenada de PNG de una carpeta resolución/clase.'''
    return sorted((DATASET_ROOT / res / cls).glob("*.png"))


def save_fig(fig, name):
    '''Guarda una figura en results/eda/ además de mostrarla inline.'''
    path = EDA_DIR / name
    fig.savefig(path, dpi=150, bbox_inches="tight")
    print(f"[figura guardada] {path}")


ModuleNotFoundError: No module named 'numpy'

## 2. Inventario y balance de clases

Cargamos el inventario desde `results/dataset_inventory.csv` si existe (generado por
`src/data/verify_dataset.py`); en caso contrario lo **recalculamos** recorriendo el
sistema de ficheros y lo persistimos. A continuación resumimos el número de imágenes
por resolución y clase, y comprobamos que el total coincide con el esperado
(**245 196** imágenes según el README oficial del dataset).

In [ ]:
inventory_csv = RESULTS_DIR / "dataset_inventory.csv"

if inventory_csv.exists():
    inv = pd.read_csv(inventory_csv)
    print("Inventario cargado desde", inventory_csv)
else:
    print("No existe results/dataset_inventory.csv — recalculando desde el sistema de ficheros...")
    rows = []
    for res in RESOLUTIONS:
        for cls in CLASSES:
            rows.append({"resolution": int(res), "class": cls, "n_images": len(list_images(res, cls))})
    inv = pd.DataFrame(rows)
    inv.to_csv(inventory_csv, index=False)
    print("Inventario guardado en", inventory_csv)

inv


In [ ]:
# Tabla resumen con totales y porcentaje de la clase minoritaria (Abnormal)
pivot = inv.pivot(index="resolution", columns="class", values="n_images")[["Normal", "Abnormal"]]
pivot["Total"] = pivot["Normal"] + pivot["Abnormal"]
pivot["%Abnormal"] = (pivot["Abnormal"] / pivot["Total"] * 100).round(1)

total_row = pd.DataFrame({
    "Normal":    [pivot["Normal"].sum()],
    "Abnormal":  [pivot["Abnormal"].sum()],
    "Total":     [pivot["Total"].sum()],
    "%Abnormal": [round(pivot["Abnormal"].sum() / pivot["Total"].sum() * 100, 1)],
}, index=["Total"])
inv_table = pd.concat([pivot, total_row])

total_images = int(inv["n_images"].sum())
print(f"Total de imágenes: {total_images} (esperado {EXPECTED_TOTAL})")
if total_images != EXPECTED_TOTAL:
    print("[AVISO] El total NO coincide con el esperado del README.")
else:
    print("[OK] El total coincide con el README.")
inv_table


In [ ]:
# Gráfico de barras del balance de clases por resolución
plot_df = inv.pivot(index="resolution", columns="class", values="n_images")[["Normal", "Abnormal"]]

fig, ax = plt.subplots(figsize=(8, 5))
plot_df.plot(kind="bar", ax=ax, color=[CLASS_COLORS["Normal"], CLASS_COLORS["Abnormal"]])
ax.set_title("Balance de clases por resolución — GasHisSDB")
ax.set_xlabel("Resolución de parche (px)")
ax.set_ylabel("Número de imágenes")
ax.legend(title="Clase")
for c in ax.containers:
    ax.bar_label(c, fmt="%d", fontsize=8, padding=2)
plt.xticks(rotation=0)
save_fig(fig, "02_balance_clases.png")
plt.show()


**Lectura:** el dataset presenta un **desbalance moderado y consistente**: la clase
`Normal` es mayoritaria (~60 %) frente a `Abnormal` (~38–40 %) en las tres resoluciones.
No es un desbalance extremo, pero es suficiente para que la *accuracy* simple sea
engañosa (un clasificador trivial que prediga siempre `Normal` alcanzaría ~60 %). Esto
motiva el uso de **métricas sensibles al desbalance** (sensibilidad/recall de `Abnormal`,
F1, AUC-PR, matriz de confusión) y, opcionalmente, **class weights** o *augmentation* de
la clase minoritaria.

## 3. Inspección visual

Mostramos una cuadrícula 4×4 de imágenes aleatorias (semilla 42) para cada una de las
6 combinaciones resolución×clase. El objetivo es familiarizarse con la apariencia del
tejido, la variabilidad de tinción H&E y posibles artefactos (parches desenfocados,
fondo sin tejido, etc.).

In [ ]:
def show_grid(res, cls, n=16, ncols=4):
    files  = list_images(res, cls)
    sample = random.Random(SEED).sample(files, min(n, len(files)))
    nrows  = (len(sample) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 2.2, nrows * 2.2))
    axes = np.array(axes).reshape(-1)
    for ax in axes:
        ax.axis("off")
    for ax, f in zip(axes, sample):
        ax.imshow(Image.open(f).convert("RGB"))
        ax.set_title(f.name, fontsize=6)
    fig.suptitle(f"Ejemplos aleatorios — {res}px / {cls} (n={len(sample)})", y=1.0, fontsize=11)
    fig.tight_layout()
    save_fig(fig, f"03_muestras_{res}_{cls}.png")
    plt.show()


In [ ]:
show_grid("80", "Normal")

In [ ]:
show_grid("80", "Abnormal")

In [ ]:
show_grid("120", "Normal")

In [ ]:
show_grid("120", "Abnormal")

In [ ]:
show_grid("160", "Normal")

In [ ]:
show_grid("160", "Abnormal")

## 4. Estadísticas de intensidad y color

Sobre una muestra de **5 000 imágenes por resolución** (estratificada por clase según su
proporción real) calculamos la **media y desviación típica por canal RGB** (en `[0, 1]`).

Dos objetivos:

1. **Detección de atajos espurios**: si las clases `Normal` y `Abnormal` difieren de forma
   sistemática en color/iluminación (p. ej. por lotes de tinción distintos), el modelo
   podría discriminar por el color global del parche en lugar de por la morfología
   histológica — un sesgo peligroso que no generalizaría a otro laboratorio.
2. **Estadísticas de normalización específicas del dominio**: las guardamos en
   `configs/normalization_stats.yaml` para usarlas en `transforms.Normalize` durante el
   entrenamiento, en lugar de los valores por defecto de ImageNet (imágenes naturales,
   dominio muy distinto al de la tinción H&E).

In [ ]:
SAMPLE_STATS  = 5000
stats_records = []   # media por imagen y canal, etiquetada por clase (para comparar clases)
norm_stats    = {}   # media/std global por resolución (para normalización)

rng = random.Random(SEED)
for res in RESOLUTIONS:
    counts = {cls: len(list_images(res, cls)) for cls in CLASSES}
    total  = sum(counts.values())
    ch_sum, ch_sumsq, npx = np.zeros(3), np.zeros(3), 0

    for cls in CLASSES:
        k       = int(round(SAMPLE_STATS * counts[cls] / total))
        sample  = rng.sample(list_images(res, cls), min(k, counts[cls]))
        for f in sample:
            arr  = np.asarray(Image.open(f).convert("RGB"), dtype=np.float64) / 255.0
            flat = arr.reshape(-1, 3)
            stats_records.append({"resolution": int(res), "class": cls,
                                  "R": flat[:, 0].mean(), "G": flat[:, 1].mean(), "B": flat[:, 2].mean()})
            ch_sum   += flat.sum(axis=0)
            ch_sumsq += (flat ** 2).sum(axis=0)
            npx      += flat.shape[0]

    mean = ch_sum / npx
    std  = np.sqrt(np.maximum(ch_sumsq / npx - mean ** 2, 0))
    norm_stats[res] = {"mean": [round(float(x), 4) for x in mean],
                       "std":  [round(float(x), 4) for x in std]}
    print(f"[{res}px] mean={norm_stats[res]['mean']}  std={norm_stats[res]['std']}")

stats_df = pd.DataFrame(stats_records)
print("Imágenes muestreadas en total:", len(stats_df))
stats_df.head()


In [ ]:
# Comparación de intensidad media por clase y canal
comp = (stats_df.groupby(["resolution", "class"])[["R", "G", "B"]]
        .agg(["mean", "std"]).round(4))
comp


In [ ]:
# Diferencia de intensidad media Abnormal - Normal (¿sesgo sistemático de color?)
delta_rows = []
for res in RESOLUTIONS:
    sub = stats_df[stats_df.resolution == int(res)]
    mn  = sub[sub["class"] == "Normal"][["R", "G", "B"]].mean()
    ma  = sub[sub["class"] == "Abnormal"][["R", "G", "B"]].mean()
    d   = ma - mn
    delta_rows.append({"resolution": int(res),
                       "dR (Abn-Nor)": round(d.R, 4),
                       "dG (Abn-Nor)": round(d.G, 4),
                       "dB (Abn-Nor)": round(d.B, 4)})
delta_df = pd.DataFrame(delta_rows)
delta_df


In [ ]:
# Boxplots de la intensidad media por canal, separando Normal vs Abnormal
melted = stats_df.melt(id_vars=["resolution", "class"], value_vars=["R", "G", "B"],
                       var_name="canal", value_name="intensidad_media")

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
for ax, res in zip(axes, RESOLUTIONS):
    sub = melted[melted["resolution"] == int(res)]
    sns.boxplot(data=sub, x="canal", y="intensidad_media", hue="class",
                palette=CLASS_COLORS, ax=ax, showfliers=False)
    ax.set_title(f"{res}px")
    ax.set_xlabel("Canal RGB")
    ax.set_ylabel("Intensidad media (0-1)")
    ax.legend(title="Clase", loc="lower center")
fig.suptitle("Distribución de intensidad media por canal — Normal vs Abnormal", y=1.02)
fig.tight_layout()
save_fig(fig, "04_intensidad_por_clase.png")
plt.show()


In [ ]:
# Persistir estadísticas de normalización específicas del dominio (una entrada por resolución)
norm_out = {}
for res in RESOLUTIONS:
    norm_out[f"gashisdb_{res}"] = {
        "resolution":      int(res),
        "mean":            norm_stats[res]["mean"],
        "std":             norm_stats[res]["std"],
        "n_sample_images": SAMPLE_STATS,
        "note": ("Media/std por canal RGB en [0,1], sobre muestra estratificada por clase "
                 "(seed 42). Usar en transforms.Normalize en lugar de los valores de ImageNet."),
    }

out_path = CONFIGS_DIR / "normalization_stats.yaml"
with open(out_path, "w", encoding="utf-8") as fh:
    yaml.safe_dump(norm_out, fh, sort_keys=False, allow_unicode=True)

print("Guardado:", out_path)
print("-" * 60)
print(out_path.read_text(encoding="utf-8"))


**Lectura:** compárese la magnitud de las diferencias `Abnormal − Normal` (tabla de
deltas) con la dispersión intra-clase (boxplots). Diferencias de color pequeñas frente a
la varianza intra-clase indican que **el color global no separa trivialmente las clases**
(bueno: reduce el riesgo de atajo espurio por tinción). Diferencias grandes y sistemáticas
serían una señal de alarma a controlar con *stain normalization*/augmentation de color.
Las estadísticas de normalización guardadas difieren notablemente de las de ImageNet
(`mean=[0.485, 0.456, 0.406]`), lo que justifica usar las propias del dominio.

## 5. Duplicados y *near-duplicates* (perceptual hashing)

Calculamos el *perceptual hash* (`imagehash.phash`, 64 bits) sobre una muestra de
**10 000 imágenes por resolución** (estratificada por clase) y buscamos pares con
**distancia de Hamming ≤ 5** (casi idénticos) y `= 0` (duplicados exactos perceptuales).
La comparación por pares se vectoriza con NumPy (XOR + *popcount* por tabla de bytes) para
que sea viable sobre ~10 000 elementos.

Duplicados abundantes serían problemáticos: si copias del mismo parche caen en *train* y en
*test*, el rendimiento reportado quedaría inflado.

In [ ]:
dup_df = None
if not HAS_IMAGEHASH:
    print("imagehash no disponible — sección omitida.")
else:
    SAMPLE_HASH = 10000
    THRESHOLD   = 5
    POP = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint16)  # popcount por byte

    rng = random.Random(SEED)
    dup_summary = []
    for res in RESOLUTIONS:
        counts = {cls: len(list_images(res, cls)) for cls in CLASSES}
        total  = sum(counts.values())
        sample = []
        for cls in CLASSES:
            k = int(round(SAMPLE_HASH * counts[cls] / total))
            sample += rng.sample(list_images(res, cls), min(k, counts[cls]))

        # phash -> 8 bytes por imagen
        hb = np.zeros((len(sample), 8), dtype=np.uint8)
        for i, f in enumerate(sample):
            h = imagehash.phash(Image.open(f).convert("RGB"))
            hb[i] = np.packbits(h.hash.flatten())

        n = len(sample)
        involved = np.zeros(n, dtype=bool)
        exact_pairs = near_pairs = 0
        for i in range(n - 1):
            dist = POP[hb[i + 1:] ^ hb[i]].sum(axis=1)   # Hamming vs el resto
            near = dist <= THRESHOLD
            exact_pairs += int((dist == 0).sum())
            nn = int(near.sum())
            near_pairs  += nn
            if nn:
                involved[i] = True
                involved[np.nonzero(near)[0] + i + 1] = True

        dup_summary.append({"resolution": int(res), "n_muestra": n,
                            "pares_exactos": exact_pairs,
                            "pares_near(<=5)": near_pairs,
                            "img_implicadas": int(involved.sum()),
                            "%_muestra": round(100 * involved.sum() / n, 2)})
        print(f"[{res}px] n={n}  exactos={exact_pairs}  near={near_pairs}  "
              f"implicadas={int(involved.sum())} ({100 * involved.sum() / n:.2f}%)")

    dup_df = pd.DataFrame(dup_summary)
    dup_df


In [ ]:
if dup_df is not None:
    fig, ax = plt.subplots(figsize=(7, 4))
    sns.barplot(data=dup_df, x="resolution", y="%_muestra", ax=ax, color="#8172B3")
    ax.set_title("Muestra implicada en near-duplicates (Hamming ≤ 5)")
    ax.set_xlabel("Resolución (px)")
    ax.set_ylabel("% de la muestra")
    ax.bar_label(ax.containers[0], fmt="%.2f%%")
    save_fig(fig, "05_duplicados.png")
    plt.show()


**Lectura:** una proporción baja de *near-duplicates* dentro de una muestra aleatoria
es esperable (parches vecinos de la misma región tisular). Lo relevante para la validez es
que estos duplicados **no crucen la frontera train/test** — algo que un *split* aleatorio a
nivel de parche no garantiza (véase la sección 6).

## 6. Análisis de posible *leakage* por WSI  ⚠️ (punto crítico)

En histopatología de parches, el riesgo metodológico más grave es la **fuga de datos a
nivel de WSI**: si parches recortados de la *misma* imagen de portaobjetos (mismo
paciente/lote de tinción) se reparten entre *train* y *test*, el modelo puede memorizar
textura/tinción local en lugar de aprender malignidad generalizable, inflando las métricas.

Para poder hacer un *split* que **no** mezcle WSIs entre particiones, necesitaríamos un
identificador de WSI/paciente por parche. Aquí examinamos los **nombres de archivo** en
busca de ese identificador.

In [ ]:
# Muestra de nombres de archivo por carpeta
for res in RESOLUTIONS:
    for cls in CLASSES:
        files  = list_images(res, cls)
        sample = random.Random(SEED).sample(files, min(6, len(files)))
        print(f"--- {res}px / {cls}  (n={len(files)}) ---")
        for f in sample:
            print("   ", f.name)


In [ ]:
# Estructura del nombre: prefijo alfabético + índice numérico
pattern = re.compile(r"^([A-Za-z]+)[-_]?(\d+)$")

def parse_name(name):
    m = pattern.match(Path(name).stem)
    return (m.group(1), int(m.group(2))) if m else (Path(name).stem, None)

struct_rows = []
for res in RESOLUTIONS:
    for cls in CLASSES:
        names    = [f.name for f in list_images(res, cls)]
        prefixes = sorted({parse_name(n)[0] for n in names})
        n_tokens = sorted({len(re.split(r"[-_]", Path(n).stem)) for n in names})
        struct_rows.append({"resolution": int(res), "class": cls,
                            "prefijos_distintos": prefixes,
                            "n_tokens_por_nombre": n_tokens,
                            "ejemplo": names[0]})
pd.DataFrame(struct_rows)


In [ ]:
# ¿Los índices forman UNA sola secuencia contigua 1..N por clase?
# (si es así, no hay agrupación por WSI recuperable del nombre)
seq_rows = []
for res in RESOLUTIONS:
    for cls in CLASSES:
        idxs = sorted(parse_name(f.name)[1] for f in list_images(res, cls))
        seq_rows.append({"resolution": int(res), "class": cls,
                         "n": len(idxs), "idx_min": idxs[0], "idx_max": idxs[-1],
                         "contigua_1..N": idxs == list(range(1, len(idxs) + 1))})
pd.DataFrame(seq_rows)


### Conclusión sobre el *leakage* por WSI

Los nombres de archivo siguen el patrón **`<Clase>-<índice>`** (p. ej. `Abnormal-00001.png`),
donde el índice es una **numeración correlativa única por clase**, sin ningún token que
identifique la WSI, el paciente o el lote de tinción de origen.

**Por tanto, no es posible reconstruir la agrupación por WSI a partir de los datos
disponibles.** Esto se declara explícitamente como **limitación metodológica documentada**
(no se asume ausencia de *leakage* solo porque no se detecte desde el nombre: los parches
provienen necesariamente de un número finito de WSIs, cuya identidad está perdida en el
*release* público del dataset).

**Implicaciones para el *split* de la Fase 1:** el README oficial de GasHisSDB prescribe un
*split* aleatorio **40/40/20** (train/val/test) *a nivel de parche*. Con la información
disponible, ese es el protocolo reproducible frente a la literatura, pero **arrastra un
riesgo de fuga por WSI que no puede eliminarse**. Se recomienda:

- Usar el *split* 40/40/20 a nivel de parche para **comparabilidad** con el estado del arte,
  reportando los resultados como **cota superior optimista**.
- **Documentar la limitación** en la memoria y en la discusión de resultados.
- Si se obtuvieran metadatos de WSI (autores del dataset / *release* ampliado), rehacer el
  *split* de forma **disjunta por WSI** como análisis de robustez.
- Reforzar el análisis de errores con **Grad-CAM** para verificar que el modelo atiende a
  morfología y no a textura de fondo (indicio de *shortcut learning* por fuga).

## 7. Consistencia entre resoluciones (80 / 120 / 160)

Comprobamos si las carpetas `80`, `120` y `160` son la **misma fuente de tejido recortada
a distinta escala** o conjuntos independientes, comparando las convenciones de nombres y la
proporción de clases.

In [ ]:
conv_rows = []
for res in RESOLUTIONS:
    for cls in CLASSES:
        files = list_images(res, cls)
        conv_rows.append({"resolution": int(res), "class": cls, "n": len(files),
                          "primer_fichero": files[0].name, "ultimo_fichero": files[-1].name})
conv_df = pd.DataFrame(conv_rows)
conv_df


In [ ]:
# Proporción de Abnormal por resolución: si es estable, apunta a la misma fuente re-escalada
frac    = inv.pivot(index="resolution", columns="class", values="n_images")
frac_ab = (frac["Abnormal"] / (frac["Abnormal"] + frac["Normal"]) * 100)

fig, ax = plt.subplots(figsize=(7, 4))
frac_ab.plot(marker="o", ax=ax, color="#C44E52")
ax.set_title("Proporción de Abnormal por resolución")
ax.set_xlabel("Resolución (px)")
ax.set_ylabel("% Abnormal")
ax.set_xticks(frac_ab.index)
for x, y in zip(frac_ab.index, frac_ab.values):
    ax.annotate(f"{y:.1f}%", (x, y), textcoords="offset points", xytext=(0, 6), ha="center")
save_fig(fig, "07_proporcion_abnormal.png")
plt.show()


**Lectura:** las tres resoluciones comparten la **misma convención de nombres**
(`<Clase>-<índice>`) y una **proporción de clases muy similar**, mientras que el número de
parches **decrece** al aumentar el tamaño (caben menos parches grandes por WSI). Todo ello
es consistente con que **80/120/160 son la misma fuente de WSIs recortada a tres escalas**,
no conjuntos independientes.

**Consecuencia metodológica:** las tres resoluciones **no deben combinarse como si fueran
datos independientes** (duplicaría información y agravaría el *leakage*). Cada resolución se
trata como un experimento separado; comparar entre resoluciones es un análisis de
*generalización cross-scale*, no una ampliación del tamaño muestral.

## 8. Conclusiones del EDA

**(a) Balance de clases.** Desbalance moderado y estable (`Normal` ~60 % / `Abnormal`
~38–40 % en las tres resoluciones). No es extremo, pero exige **no** usar *accuracy* como
métrica principal. Recomendación: métricas sensibles al desbalance (sensibilidad/recall de
`Abnormal`, F1, AUC-PR, matriz de confusión) y, como manejo, **class weights** en la pérdida
y/o *augmentation* de la clase minoritaria. No es imprescindible sobre-muestrear de forma
agresiva dado el desbalance contenido.

**(b) Leakage por WSI — hallazgo crítico.** Los nombres de archivo (`<Clase>-<índice>`)
**no contienen identificador de WSI/paciente**, y los índices son una secuencia correlativa
única por clase. Es **imposible** reconstruir la agrupación por WSI. El *split* aleatorio
40/40/20 a nivel de parche del README oficial se usará por **comparabilidad**, pero
**arrastra riesgo de fuga inevitable**: los resultados deben reportarse como **cota superior
optimista** y la limitación documentarse explícitamente. Verificar con Grad-CAM que el
modelo no aprende atajos de textura/fondo.

**(c) Duplicados.** El *perceptual hashing* (muestra de 10 000/resolución, Hamming ≤ 5)
cuantifica *near-duplicates*; véase la tabla de la sección 5 para las cifras exactas. Una
proporción baja es esperable por vecindad tisular; el problema no es su existencia sino que
crucen la frontera train/test — riesgo ligado al mismo problema de *split* del punto (b).

**(d) Color/tinción entre clases.** Las diferencias de intensidad media `Abnormal − Normal`
por canal (sección 4) deben compararse con la dispersión intra-clase: si son pequeñas, el
color global **no** separa trivialmente las clases (menor riesgo de atajo espurio por
tinción). Se han calculado y guardado **estadísticas de normalización propias del dominio**
en `configs/normalization_stats.yaml`, que difieren de las de ImageNet y deben usarse en el
pipeline de entrenamiento.

---
*Artefactos generados por este cuaderno:* `results/dataset_inventory.csv`,
`results/eda/*.png`, `configs/normalization_stats.yaml`.
